In [45]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

In [46]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from lightgbm import LGBMRegressor

from sklearn.pipeline import Pipeline

from src.data_loader import load_data
from src.task2.feature_engineering import FeatureStore
from src.task2.evaluator import evaluate_model

In [47]:
MODEL_PATH = PROJECT_ROOT / 'src' / 'task2' / 'duration_fcst_pipe.pkl'

In [48]:
data = load_data('data_2')

In [54]:
# Target
data['duration'] = (data.time_to - data.time_from).dt.total_seconds()

# Rolling-origin CV
data['date'] = data.time_from.dt.normalize()
date_split = data['date'].max() - pd.Timedelta(days=7)

cv_metrics_mae = []

for i in range(7):
    test_start = date_split + pd.Timedelta(days=i)
    test_end = test_start + pd.Timedelta(days=1)
    
    train_set = data[data['date'] < test_start]
    test_set = data[(data['date'] >= test_start) & (data['date'] < test_end)]

    print(f'Run {i} Train: {train_set.time_from.min(), train_set.time_from.max()} Test: {test_set.time_from.min(), test_set.time_from.max()}')
    
    if test_set.empty:
        continue
        
    fe_transformer = FeatureStore()
    
    X_train = fe_transformer.fit_transform(train_set)
    y_train = train_set['duration']
    
    X_test = fe_transformer.transform(test_set)
    y_test = test_set['duration']

    params = {
        'n_estimators': 150,
        'min_child_samples': 50,
        'num_leaves': 15,
        'max_depth': 10,
        'reg_alpha': 0.5,
        'reg_lambda': 0.5,
        'verbose': -1
    }

    model = LGBMRegressor(**params)
    model.fit(X_train, y_train)
    
    print(f'Fold {i+1} Date: {test_start.date()} Test Samples: {len(test_set)}')
    results = evaluate_model(model, X_test, y_test)

    print('Train:')
    _ = evaluate_model(model, X_train, y_train)

    print('-' * 50)
    
    cv_metrics_mae.append(results['mae'])

print(f"Average MAE {np.mean(cv_metrics_mae):.2f}")

Run 0 Train: (Timestamp('2025-07-01 08:06:01'), Timestamp('2025-09-22 22:43:32')) Test: (Timestamp('2025-09-23 08:08:11'), Timestamp('2025-09-23 22:56:59'))
Fold 1 Date: 2025-09-23 Test Samples: 41
MAE: 4.97 RMSE: 6.63 R2 score: 0.55
Train:
MAE: 3.82 RMSE: 4.80 R2 score: 0.84
--------------------------------------------------
Run 1 Train: (Timestamp('2025-07-01 08:06:01'), Timestamp('2025-09-23 22:56:59')) Test: (Timestamp('2025-09-24 08:01:36'), Timestamp('2025-09-24 22:47:42'))
Fold 2 Date: 2025-09-24 Test Samples: 45
MAE: 7.06 RMSE: 8.60 R2 score: 0.20
Train:
MAE: 3.81 RMSE: 4.79 R2 score: 0.84
--------------------------------------------------
Run 2 Train: (Timestamp('2025-07-01 08:06:01'), Timestamp('2025-09-24 22:47:42')) Test: (Timestamp('2025-09-25 08:42:14'), Timestamp('2025-09-25 22:43:54'))
Fold 3 Date: 2025-09-25 Test Samples: 53
MAE: 6.25 RMSE: 7.67 R2 score: 0.60
Train:
MAE: 3.81 RMSE: 4.80 R2 score: 0.84
--------------------------------------------------
Run 3 Train: (Ti

In [65]:
# Train final model for inference
final_fe_transformer = FeatureStore()

final_train = data[data.date < data.date.max()]
final_test = data[data.date >= data.date.max()]

X_final_train = final_fe_transformer.fit_transform(final_train)
y_final_train = final_train['duration']

X_final_test = final_fe_transformer.transform(final_test)
y_final_test = final_test['duration']

final_model = LGBMRegressor(**params)
final_model.fit(X_final_train, y_final_train)

evaluate_model(model, X_final_test, y_final_test)

pipe = Pipeline([
    ('feature_engineering', final_fe_transformer),
    ('lightgbm_model', final_model)
])

MAE: 5.74 RMSE: 7.06 R2 score: 0.46


In [66]:
# Save pipiline with feature engineering and model for inference
with MODEL_PATH.open('wb') as f:
    pickle.dump(pipe, f)